# 🤖 Asistente RRHH Falabella - Pipeline RAG
## ISY0101 - Ingeniería de Soluciones con IA
### Integrantes: Kevin Morales Riveros - Matias Damico

Este notebook implementa un pipeline RAG (Retrieval-Augmented Generation) para responder preguntas de empleados de Falabella sobre temas de Recursos Humanos, basándose en documentos internos oficiales.


## 📦 Paso 1: Instalación de dependencias

In [ ]:
# Instalación de librerías necesarias
!pip install langchain langchain-openai langchain-community chromadb openai python-dotenv langchain-text-splitters -q
print("✅ Dependencias instaladas correctamente")

## 🔑 Paso 2: Configuración de credenciales
Ingresa tu GitHub Token para usar GitHub Models (gratuito).


In [ ]:
import os

# Ingresa tu GitHub Token aquí
GITHUB_TOKEN = "TU_GITHUB_TOKEN_AQUI"  # <-- Reemplaza con tu token
BASE_URL = "https://models.inference.ai.azure.com"

os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ["OPENAI_API_KEY"] = GITHUB_TOKEN
os.environ["OPENAI_BASE_URL"] = BASE_URL

print("✅ Credenciales configuradas correctamente")

## 📂 Paso 3: Creación de documentos internos de RRHH
Creamos los documentos simulados de Falabella que servirán como base de conocimiento del agente.


In [ ]:
# Crear documentos simulados de RRHH de Falabella
import os

os.makedirs("docs_rrhh", exist_ok=True)

reglamento = """REGLAMENTO INTERNO DE ORDEN, HIGIENE Y SEGURIDAD
FALABELLA RETAIL S.A. - Versión 2024

1. JORNADA DE TRABAJO
La jornada ordinaria es de 45 horas semanales.
Personal de tienda: turnos rotativos entre 08:00 y 22:00 horas.
Personal administrativo: lunes a viernes de 09:00 a 18:30 horas.
Personal de bodega: turnos de 06:00 a 14:00 o de 14:00 a 22:00 horas.
Las horas extraordinarias se pagan con recargo del 50%.

2. REMUNERACIONES
Las remuneraciones se pagan el último día hábil de cada mes mediante depósito bancario.
El trabajador puede acceder a su liquidación desde el día 25 de cada mes en https://empleados.falabella.com.
Se pueden solicitar anticipos de hasta el 50% del sueldo si se llevan al menos 3 meses en la empresa.

3. CONDUCTA Y DISCIPLINA
Los trabajadores deben mantener un trato respetuoso con compañeros, clientes y proveedores.
Queda prohibido el acoso laboral o sexual.
Canal de denuncias: etica@falabella.com | 600 390 0000

4. HIGIENE Y SEGURIDAD
En caso de accidente, informar de inmediato al supervisor y será derivado a la ACHS.
El uso de EPP es obligatorio en bodegas y centros de distribución.

5. CONFIDENCIALIDAD
Todos los trabajadores deben mantener reserva sobre información comercial, financiera y de clientes.
"""

beneficios = """MANUAL DE BENEFICIOS PARA COLABORADORES
FALABELLA RETAIL S.A. - Versión 2024

1. BENEFICIOS DE SALUD
Seguro Complementario de Salud para trabajadores con contrato indefinido y 6 meses de antigüedad.
Coberturas: Hospitalización hasta UF 300, atención ambulatoria 80%, medicamentos 60%, dental $300.000 anuales.
Convenio con red de médicos con descuentos de hasta 20%.

2. BENEFICIOS ECONÓMICOS
Bono de desempeño anual: entre 1 y 3 sueldos base, pagado en marzo.
Aguinaldo Fiestas Patrias: $80.000 (primera quincena de septiembre).
Aguinaldo Navidad: $100.000 (primera quincena de diciembre).
Descuento del 15% en compras en Falabella, Sodimac y Tottus con credencial de empleado.
Préstamos de hasta 3 sueldos base para trabajadores con 1 año de antigüedad.

3. BIENESTAR
Caja de Compensación Los Andes con créditos y actividades recreativas.
Bono por matrimonio o unión civil: $150.000.
Bono por nacimiento de hijo: $100.000.
Día libre en el mes de cumpleaños (coordinar con jefe directo con 5 días de anticipación).
Programa de Asistencia al Empleado (PAE): orientación psicológica, legal y financiera gratuita.
Contacto PAE: pae@falabella.com | 600 390 2222 (24/7)
"""

vacaciones = """POLÍTICA DE VACACIONES, PERMISOS Y LICENCIAS
FALABELLA RETAIL S.A. - Versión 2024

1. VACACIONES ANUALES
Trabajadores con 1 a 10 años de antigüedad: 15 días hábiles de vacaciones.
Trabajadores con más de 10 años en Falabella: 20 días hábiles (beneficio adicional).
Las vacaciones deben solicitarse con 15 días de anticipación en https://empleados.falabella.com.
En noviembre, diciembre y CyberDay las vacaciones están sujetas a restricciones operativas.

2. PERMISOS
Por fallecimiento de cónyuge, hijos o padre/madre: 7 días hábiles pagados.
Por fallecimiento de hermanos, abuelos o suegros: 3 días hábiles pagados.
Por matrimonio o unión civil: 5 días hábiles pagados.
Por nacimiento de hijo (padre): 5 días hábiles pagados.
Día libre para votar en elecciones sin descuento en remuneración.

3. LICENCIAS MÉDICAS
Enviar la licencia a RRHH dentro de 48 horas de emitida.
Correo: licencias@falabella.com o a través del portal de empleados.
Los accidentes de trabajo son cubiertos por la ACHS.

4. PRE Y POSTNATAL
Prenatal: 6 semanas antes del parto.
Postnatal: 12 semanas básicas + 12 semanas de postnatal parental.
"""

faq = """PREGUNTAS FRECUENTES - RECURSOS HUMANOS
FALABELLA RETAIL S.A.

P: ¿Cuándo se paga el sueldo?
R: El sueldo se paga el último día hábil de cada mes mediante depósito bancario.

P: ¿Cómo accedo a mi liquidación?
R: En https://empleados.falabella.com sección 'Mis Documentos', disponible desde el día 25.

P: ¿Cuántos días de vacaciones me corresponden?
R: Entre 1 y 10 años de antigüedad: 15 días hábiles. Más de 10 años en Falabella: 20 días hábiles.

P: ¿Tengo descuento en tiendas Falabella?
R: Sí, 15% en Falabella, Sodimac y Tottus con credencial de empleado.

P: ¿Qué hago si tuve un accidente en el trabajo?
R: Informar de inmediato al supervisor. Serás derivado a la ACHS. No abandones el lugar sin reportarlo.

P: ¿Puedo tomar vacaciones en diciembre?
R: En noviembre, diciembre y CyberDay las vacaciones están sujetas a restricciones y requieren aprobación de Gerencia.

P: ¿Cuándo se pagan los aguinaldos?
R: Fiestas Patrias en septiembre ($80.000) y Navidad en diciembre ($100.000), con 6 meses de antigüedad mínimo.

P: ¿Cuántos días de permiso si fallece un familiar?
R: Cónyuge, hijos o padres: 7 días hábiles. Hermanos, abuelos, suegros: 3 días hábiles.
"""

# Guardar documentos
with open("docs_rrhh/reglamento_interno.txt", "w", encoding="utf-8") as f:
    f.write(reglamento)
with open("docs_rrhh/manual_beneficios.txt", "w", encoding="utf-8") as f:
    f.write(beneficios)
with open("docs_rrhh/politica_vacaciones_permisos.txt", "w", encoding="utf-8") as f:
    f.write(vacaciones)
with open("docs_rrhh/faq_rrhh.txt", "w", encoding="utf-8") as f:
    f.write(faq)

print("✅ Documentos de RRHH creados:")
for archivo in os.listdir("docs_rrhh"):
    print(f"  📄 {archivo}")

## ⚙️ Paso 4: Implementación del Pipeline RAG
Cargamos los documentos, creamos los embeddings y configuramos el retriever.


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── Cargar documentos ──
print("📂 Cargando documentos de RRHH...")
documentos_rrhh = [
    "docs_rrhh/reglamento_interno.txt",
    "docs_rrhh/manual_beneficios.txt",
    "docs_rrhh/politica_vacaciones_permisos.txt",
    "docs_rrhh/faq_rrhh.txt"
]
documentos = []
for archivo in documentos_rrhh:
    loader = TextLoader(archivo, encoding="utf-8")
    docs = loader.load()
    documentos.extend(docs)
    print(f"  ✅ {archivo} cargado")

# ── Dividir en chunks ──
print("\n✂️ Dividiendo en fragmentos...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_documents(documentos)
print(f"✅ {len(chunks)} fragmentos creados")

# ── Crear embeddings y base vectorial ──
print("\n🔢 Creando embeddings y base de datos vectorial...")
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=GITHUB_TOKEN,
    openai_api_base=BASE_URL
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print("✅ Base de datos vectorial creada")

# ── Configurar retriever ──
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("✅ Retriever configurado")

## 💬 Paso 5: Configuración de Prompts y LLM
Definimos los prompts optimizados y conectamos el modelo de lenguaje.


In [ ]:
# ── Prompt optimizado (Zero-shot con rol) ──
prompt_template = """
Eres un asistente virtual de Recursos Humanos de Falabella Retail S.A.
Tu rol es responder preguntas de los empleados de manera clara, precisa y amable.

Reglas importantes:
- Responde ÚNICAMENTE basándote en la información de los documentos proporcionados.
- Si la información no está en los documentos, responde: "Lo siento, no tengo información
  sobre ese tema. Te recomiendo contactar directamente al equipo de RRHH al 600 390 1111."
- Usa un tono profesional pero cercano.
- Si la respuesta incluye números, fechas o montos, cítalos exactamente como aparecen.
- No inventes información que no esté en los documentos.

Contexto de los documentos:
{context}

Pregunta del empleado: {question}

Respuesta:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# ── Configurar LLM (GitHub Models - Gratuito) ──
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    openai_api_key=GITHUB_TOKEN,
    openai_api_base=BASE_URL
)

# ── Crear cadena RAG ──
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG completo configurado correctamente")

## 🧪 Paso 6: Pruebas del sistema
Ejecutamos 5 preguntas de prueba para verificar el funcionamiento del asistente.


In [ ]:
def consultar_rrhh(pregunta):
    print(f"\n{'='*60}")
    print(f"👤 Empleado pregunta: {pregunta}")
    print(f"{'='*60}")
    respuesta = rag_chain.invoke(pregunta)
    print(f"🤖 Asistente RRHH responde:\n{respuesta}")
    return respuesta

# 5 preguntas de prueba
preguntas_prueba = [
    "¿Cuántos días de vacaciones me corresponden si llevo 5 años en la empresa?",
    "¿Cuándo se paga el sueldo cada mes?",
    "¿Tengo descuento en tiendas Falabella como empleado?",
    "¿Qué hago si tuve un accidente en el trabajo?",
    "¿Puedo tomar vacaciones en diciembre?"
]

print("🚀 INICIANDO PRUEBAS DEL ASISTENTE RRHH FALABELLA")
for pregunta in preguntas_prueba:
    consultar_rrhh(pregunta)

## 💬 Paso 7: Modo interactivo
Escribe tu propia pregunta para probar el asistente.


In [ ]:
# Cambia esta pregunta por la que quieras probar
mi_pregunta = "¿Cuántos días de permiso me dan si fallece mi papá?"

consultar_rrhh(mi_pregunta)